In [ ]:
# ==== INPUT: URLs to run through the pipeline (edit this cell, then Run All) ====
URLS = [
    "https://www.vietcombank.com.vn/",
    "http://vietcombank-xacthuc-taikhoan.com/login",
    "https://paypa1-secure-login.weebly.com/signin",
]

USE_LLM = True             # Layer D (paid OpenAI call, only for URLs that reach D)
USE_HOST_FEATURES = True   # Layer C DNS / TLS / RDAP lookups (False: URL text only, offline)
EXPLAIN = False            # ask the LLM for an explanation (extra cost)
LANG = "vi"                # explanation language
REFRESH_BLOCKLIST = False  # True: download the feeds that are due before running

# Phishing detection pipeline — demo

Runs every URL of the cell above through the production pipeline (`src/pipeline.py`, `analyze_url`):

| Layer | What it does | Stops with |
|---|---|---|
| A | Public phishing feeds (+ redirect targets), verified official domains | phishing / clean |
| B | Brand impersonation (Damerau-Levenshtein + Jaro-Winkler on the whole URL) | flag for D |
| C | BiLSTM score + XGBoost (URL + host features), DOM model when active | clean / suspicious |
| D | LLM on the sanitized page (prompt-injection filtered) | phishing / suspicious / safe |

Final verdict: `phishing`, `suspicious` (warn the user) or `clean`.

## 1. Setup

In [ ]:
import json, os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline.py").exists())
for p in (ROOT / "src", ROOT / "src" / "llm"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
pd.set_option("display.max_colwidth", 120)

if REFRESH_BLOCKLIST:
    out = subprocess.run([sys.executable, str(ROOT / "src" / "blocklist" / "updater.py")],
                         cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(out.stdout[-1500:] or out.stderr[-1500:])

from pipeline import analyze_url
from blocklist import lookup
from url_classifier import predict as classifier
from url_classifier import dom_features

# Readiness of each layer
meta = json.loads((ROOT / "data" / "blocklist" / "meta.json").read_text(encoding="utf-8"))
age_h = (datetime.now(timezone.utc) - datetime.fromisoformat(meta["updated_at"])).total_seconds() / 3600
print(f"Layer A  blocklist: {lookup.size():,} URLs, updated {age_h:.1f} h ago"
      + ("  <- stale, set REFRESH_BLOCKLIST = True" if age_h > 6 else ""))
print(f"Layer C  classifier: {'ready' if classifier.is_available() else 'NOT TRAINED'}, "
      f"DOM model: {'active' if dom_features.is_active() else 'off'}")
print(f"Layer D  LLM: {'on' if USE_LLM else 'off'}")

## 2. Run the pipeline

In [ ]:
results = []
for url in URLS:
    r = analyze_url(url, use_llm=USE_LLM, use_host_features=USE_HOST_FEATURES,
                    explain=EXPLAIN, lang=LANG)
    results.append(r)
    print(f"{r['verdict'].upper():<10} {r['latency_ms']:>8.0f} ms  {url}")

summary = pd.DataFrame([{
    "url": r["url"], "verdict": r["verdict"], "decided_by": r["decided_by"],
    "C_score": (r["layers"].get("C_classifier") or {}).get("score"),
    "latency_ms": r["latency_ms"], "explanation": r["explanation"],
} for r in results])
summary

## 3. Layer by layer — what each layer said for every URL (a layer is missing when the pipeline stopped before it).

In [ ]:
LAYER_NAMES = {"A_blocklist": "A blocklist",
               "B_typosquatting": "B typosquatting", "C_classifier": "C classifier", "D_llm": "D LLM"}

def layer_line(name, d):
    if name == "C_classifier":
        dom = f", DOM {d['dom_score']:.3f}" if d.get("dom_score") is not None else ""
        score = d.get("score")
        return (f"{d.get('label')} (score {score:.3f} / threshold {d.get('threshold', 0):.3g}{dom})"
                if score is not None else f"{d.get('label')} - {d.get('reason', '')}")
    if name == "D_llm":
        return f"{d.get('verdict')} ({d.get('latency_ms')} ms){' TIMEOUT' if d.get('timed_out') else ''}"
    return f"flag={d.get('flag')} - {d.get('reason', '')}"

for r in results:
    print()
    print(r["url"])
    print(f"  => {r['verdict'].upper()} (decided by {r['decided_by']})")
    for key, d in r["layers"].items():
        print(f"  {LAYER_NAMES.get(key, key):<17} {layer_line(key, d)}")

## 4. Raw JSON (what an API returns)

In [ ]:
print(json.dumps(results[0], ensure_ascii=False, indent=2, default=str))